In [1]:
! pip install kfp -q
! pip install gcsfs -q
! pip install google-cloud-aiplatform -q

In [2]:
#imports
import kfp, sys
from kfp.dsl import component, pipeline
from kfp import compiler
from google.cloud import aiplatform
import os
import uuid
from git import Union



In [23]:
PROJECT_ID = "apnea-detection-494320"
LOCATION = "us-central1"
BUCKET_URI = f"gs://apnea-detector-bucket"

PY_VER = f"{sys.version_info.major}.{sys.version_info.minor}"
KFP_VER = kfp.__version__   # e.g., '2.14.6'
print('Python version:  ', PY_VER)
print('KFP version:  ',KFP_VER)
aiplatform.init(project=PROJECT_ID, location=LOCATION)


Python version:   3.12
KFP version:   2.16.0


In [4]:
aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

# Real Experiment Test 

In [50]:
PROJECT_ID = "apnea-detection-494320"
LOCATION = "us-central1"
BUCKET_URI = f"gs://apnea-detector-bucket"

PY_VER = f"{sys.version_info.major}.{sys.version_info.minor}"
KFP_VER = kfp.__version__   # e.g., '2.14.6'
print('Python version:  ', PY_VER)
print('KFP version:  ',KFP_VER)
aiplatform.init(project=PROJECT_ID, location=LOCATION)


Python version:   3.12
KFP version:   2.16.0


In [53]:
# Specify a name for the experiment
EXPERIMENT_NAME = "gcp-apnea-test-experiment"

if EXPERIMENT_NAME == "[your-experiment-name]":
    EXPERIMENT_NAME = f"example-{uuid.uuid1()}"
# Create experiment
aiplatform.init(experiment=EXPERIMENT_NAME)
aiplatform.start_run("run-1")

Associating projects/257684551037/locations/us-central1/metadataStores/default/contexts/gcp-apnea-test-experiment-run-1 to Experiment: gcp-apnea-test-experiment


In [21]:
from src.features import raw_data_loader, feature_encoder, transformers

df = raw_data_loader.load_and_clean_raw("..")
df = feature_encoder.encode_features(df)

In [31]:
def create_and_import_dataset_tabular_gcs_sample(
    display_name: str,
    project: str,
    location: str,
    gcs_source: Union[str, list[str]],
):

    aiplatform.init(project=project, location=location)

    dataset = aiplatform.TabularDataset.create(
        display_name=display_name,
        gcs_source=gcs_source,
    )

    dataset.wait()

    print(f'\tDataset: "{dataset.display_name}"')
    print(f'\tname: "{dataset.resource_name}"')

I0424 15:01:28.407339 33852123 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0424 15:01:28.433948 33852123 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


In [43]:
# make 18-factor analysis version
from sklearn.model_selection import train_test_split

fa_transformer = transformers.Factor_Analyzer_Transformer(n_factors=18, rotation="varimax")

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["ahi"]), df["ahi"], test_size=0.2, random_state=42
)

fa_transformer.fit(X_train)

df_transformed = fa_transformer.transform(df.drop(columns=["ahi"]))
df_transformed["ahi"] = df["ahi"]


In [44]:
df_transformed.to_csv("gs://apnea-detector-bucket/processed_data_fa_18.csv", index=False)

In [55]:
# Register source dataset
raw_artifact = aiplatform.Artifact.create(
    schema_title="system.Dataset",
    display_name="apnea_dataset-1.0",
    uri="gs://apnea-detector-bucket/processed_data.csv",
)

# Register derived dataset
fa_artifact = aiplatform.Artifact.create(
    schema_title="system.Dataset",
    display_name="apnea_dataset-fa-1.0",
    uri="gs://apnea-detector-bucket/processed_data_fa_18.csv",
)

# Link them via an execution (the FA transform step)
with aiplatform.start_execution(
    schema_title="system.ContainerExecution",
    display_name="factor_analysis_transform",
) as execution:
    execution.assign_input_artifacts([raw_artifact])

    # ... run your FA transform and save ...

    execution.assign_output_artifacts([fa_artifact])
    
    # Optionally log transform params
    aiplatform.log_params({"n_factors": 18, "rotation": "varimax"})

print(execution.get_output_artifacts()[0].lineage_console_uri)

https://console.cloud.google.com/vertex-ai/locations/us-central1/metadata-stores/default/artifacts/2bace3d6-0590-4f68-abe6-7c928275d7c1?project=apnea-detection-494320


In [56]:
aiplatform.end_run()


In [57]:
try:
    exp.delete()
except Exception as e:
    print(e)

Experiment run run-1 skipped backing tensorboard run deletion.
To delete backing tensorboard run, execute the following:
tensorboard_run_artifact = aiplatform.metadata.artifact.Artifact(artifact_name=f"gcp-apnea-test-experiment-run-1-tb-run")
tensorboard_run_resource = aiplatform.TensorboardRun(tensorboard_run_artifact.metadata["resourceName"])
tensorboard_run_resource.delete()
tensorboard_run_artifact.delete()
Deleting Context : projects/257684551037/locations/us-central1/metadataStores/default/contexts/gcp-apnea-test-experiment-run-1
Context deleted. . Resource name: projects/257684551037/locations/us-central1/metadataStores/default/contexts/gcp-apnea-test-experiment-run-1
Deleting Context resource: projects/257684551037/locations/us-central1/metadataStores/default/contexts/gcp-apnea-test-experiment-run-1
Delete Context backing LRO: projects/257684551037/locations/us-central1/metadataStores/default/contexts/gcp-apnea-test-experiment-run-1/operations/9011875383204642816
Context resour

In [58]:
tensorboard_run_artifact = aiplatform.metadata.artifact.Artifact(artifact_name=f"gcp-apnea-test-experiment-run-1-tb-run")
tensorboard_run_resource = aiplatform.TensorboardRun(tensorboard_run_artifact.metadata["resourceName"])
tensorboard_run_resource.delete()
tensorboard_run_artifact.delete()

Deleting TensorboardRun : projects/257684551037/locations/us-central1/tensorboards/4144527717041176576/experiments/gcp-apnea-test-experiment/runs/run-1
TensorboardRun deleted. . Resource name: projects/257684551037/locations/us-central1/tensorboards/4144527717041176576/experiments/gcp-apnea-test-experiment/runs/run-1
Deleting TensorboardRun resource: projects/257684551037/locations/us-central1/tensorboards/4144527717041176576/experiments/gcp-apnea-test-experiment/runs/run-1
Delete TensorboardRun backing LRO: projects/257684551037/locations/us-central1/tensorboards/4144527717041176576/experiments/gcp-apnea-test-experiment/operations/7952403570865733632
TensorboardRun resource projects/257684551037/locations/us-central1/tensorboards/4144527717041176576/experiments/gcp-apnea-test-experiment/runs/run-1 deleted.
Deleting Artifact : projects/257684551037/locations/us-central1/metadataStores/default/artifacts/gcp-apnea-test-experiment-run-1-tb-run
Artifact deleted. . Resource name: projects/2

# 2.3 Run — SMOTE + FA(18) + XGBoost (Binary AHI≥5)
Ports the SMOTE + FA(18) + XGBoost binary classification run from notebook 2.3 to Vertex AI Experiments.

In [59]:
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report, confusion_matrix
from src.features.transformers import Factor_Analyzer_Transformer
from src.utils.data_utils import convert_ahi

In [60]:
# Binary labels: AHI >= 5 → 1 (apnea), < 5 → 0 (no apnea)
# df, X_train, X_test, y_train, y_test are already loaded from the data cells above
y_train_bin = convert_ahi(y_train)
y_test_bin  = convert_ahi(y_test)

neg, pos = (y_train_bin == 0).sum(), (y_train_bin == 1).sum()
print(f"Train — negative: {neg}, positive: {pos}, prevalence: {pos / len(y_train_bin):.1%}")

Train — negative: 455, positive: 894, prevalence: 66.3%


In [ ]:
import gcsfs
import joblib

SMOTE_RUN_EXPERIMENT = "2-3-smote-fa-xgb-binary"
DATASET_GCS_URI = "gs://apnea-detector-bucket/processed_data_fa_18.csv"
MODEL_GCS_URI   = "gs://apnea-detector-bucket/models/smote_fa18_xgb.joblib"

aiplatform.init(project=PROJECT_ID, location=LOCATION)
aiplatform.init(experiment=SMOTE_RUN_EXPERIMENT)
aiplatform.start_run("smote-fa18-xgb-run-1")

# ── FA transform ──────────────────────────────────────────────────────────────
N_FACTORS    = 18
ROTATION     = "varimax"
N_ESTIMATORS = 250
MAX_DEPTH    = 4

fa = Factor_Analyzer_Transformer(rotation=ROTATION, n_factors=N_FACTORS)
X_train_fa = fa.fit_transform(X_train)
X_test_fa  = fa.transform(X_test)

# ── SMOTE resampling ──────────────────────────────────────────────────────────
X_resampled, y_resampled = SMOTE(random_state=42).fit_resample(X_train_fa, y_train_bin)

# ── Register dataset artifact (input) ─────────────────────────────────────────
dataset_art = aiplatform.Artifact.create(
    schema_title="system.Dataset",
    display_name="apnea_dataset-fa18",
    uri=DATASET_GCS_URI,
)

# ── Register model artifact (output) ──────────────────────────────────────────
model_art = aiplatform.Artifact.create(
    schema_title="system.Model",
    display_name="smote_fa18_xgb_model",
    uri=MODEL_GCS_URI,
)

# ── Train, evaluate, and track lineage inside a single execution ──────────────
with aiplatform.start_execution(
    schema_title="system.ContainerExecution",
    display_name="smote_fa18_xgb_training",
) as execution:
    execution.assign_input_artifacts([dataset_art])

    aiplatform.log_params({
        "pipeline":      "smote_fa_xgb",
        "n_factors":     N_FACTORS,
        "rotation":      ROTATION,
        "n_estimators":  N_ESTIMATORS,
        "max_depth":     MAX_DEPTH,
        "ahi_threshold": 5,
        "smote":         True,
    })

    clf = XGBClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=42, eval_metric="logloss")
    clf.fit(X_resampled, y_resampled)

    # Save model to GCS
    fs = gcsfs.GCSFileSystem(project=PROJECT_ID)
    with fs.open(MODEL_GCS_URI, "wb") as f:
        joblib.dump(clf, f)

    preds = clf.predict(X_test_fa)
    proba = clf.predict_proba(X_test_fa)[:, 1]

    roc_auc  = float(roc_auc_score(y_test_bin, proba))
    accuracy = float(accuracy_score(y_test_bin, preds))
    f1_macro = float(f1_score(y_test_bin, preds, average="macro"))
    f1_pos   = float(f1_score(y_test_bin, preds, pos_label=1))

    aiplatform.log_metrics({
        "roc_auc":  round(roc_auc,  4),
        "accuracy": round(accuracy, 4),
        "f1_macro": round(f1_macro, 4),
        "f1_pos":   round(f1_pos,   4),
    })

    execution.assign_output_artifacts([model_art])
    lineage_uri = execution.get_output_artifacts()[0].lineage_console_uri
    aiplatform.log_metrics({"lineage": lineage_uri})

aiplatform.end_run()

print(f"ROC-AUC:  {roc_auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 macro: {f1_macro:.4f}")
print()
print(classification_report(y_test_bin, preds, target_names=["no apnea", "apnea"]))
print(confusion_matrix(y_test_bin, preds))
print(f"\nLineage: {lineage_uri}")

Associating projects/257684551037/locations/us-central1/metadataStores/default/contexts/2-3-smote-fa-xgb-binary-smote-fa18-xgb-run-1 to Experiment: 2-3-smote-fa-xgb-binary


I0424 15:43:47.488230 33852123 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


ROC-AUC:  0.5025
Accuracy: 0.5118
F1 macro: 0.4906

              precision    recall  f1-score   support

    no apnea       0.35      0.44      0.39       119
       apnea       0.64      0.55      0.59       219

    accuracy                           0.51       338
   macro avg       0.50      0.49      0.49       338
weighted avg       0.54      0.51      0.52       338

[[ 52  67]
 [ 98 121]]


In [62]:
# Pull results from Vertex AI Experiments
experiment_df = aiplatform.get_experiment_df()
run_df = experiment_df[experiment_df.experiment_name == SMOTE_RUN_EXPERIMENT]
print(run_df.T)

                                           0
experiment_name      2-3-smote-fa-xgb-binary
run_name                smote-fa18-xgb-run-1
run_type                system.ExperimentRun
state                               COMPLETE
param.pipeline                  smote_fa_xgb
param.ahi_threshold                      5.0
param.n_factors                         18.0
param.n_estimators                     250.0
param.rotation                       varimax
param.smote                             True
param.max_depth                          4.0
metric.f1_pos                         0.5946
metric.accuracy                       0.5118
metric.f1_macro                       0.4906
metric.roc_auc                        0.5025
